In [1]:
# conda create --name bcitoolbox python=3.10


# !python mat2csv.py

In [6]:
import os

for sigma_a in [0.1, 0.3, 0.4, 0.6, 0.8]: #0.2, 
    os.system(f"python model_fitting.py -sigma_a {sigma_a} -mu_p True")
    # os.system(f"python model_fitting.py -sigma_a {sigma_a} -mu_p False")

In [5]:
from helpers import find_best_models
find_best_models('mle', sigma_a=0.2, mup=True)

                     pcommon      sigma_p      mu_p      sigma_v        \
                        mean  sem    mean  sem mean  sem    mean   sem   
group      location                                                      
Control    10            1.0  0.0  4000.0  0.0  1.5  0.0    0.30  0.02   
           15            1.0  0.0  4000.0  0.0  1.5  0.0    0.31  0.02   
           5             1.0  0.0  4000.0  0.0  1.5  0.0    0.30  0.02   
           invisible     1.0  0.0  4000.0  0.0  1.5  0.0    0.96  0.02   
           raw           1.0  0.0  4000.0  0.0  1.5  0.0    0.30  0.02   
           visible       1.0  0.0  4000.0  0.0  1.5  0.0    0.30  0.02   
Low Vision 10            1.0  0.0  4000.0  0.0  1.5  0.0    0.68  0.07   
           15            1.0  0.0  4000.0  0.0  1.5  0.0    0.74  0.07   
           5             1.0  0.0  4000.0  0.0  1.5  0.0    0.62  0.07   
           invisible     1.0  0.0  4000.0  0.0  1.5  0.0    0.94  0.04   
           raw           1.0  0.0  400

,subject_id,location,strategy,fit_type,error,bic,r2,pcommon,sigma_v,sigma_a,sigma_p,mu_p,group
0,LV001,raw,Averaging,mll,1161.670578,2330.045570,0.334220,1.0,0.983829,0.2,4000,1.5,Low Vision
1,LV002,raw,Averaging,mll,481.038757,968.781929,0.884695,1.0,0.997635,0.2,4000,1.5,Low Vision
2,LV003,raw,Averaging,mll,489.144638,984.993689,0.922049,1.0,0.965402,0.2,4000,1.5,Low Vision
3,LV005,raw,Averaging,mll,671.096367,1348.897148,0.796951,1.0,0.878501,0.2,4000,1.5,Low Vision
4,LV007,raw,Averaging,mll,182.131171,370.966757,0.960125,1.0,0.269074,0.2,4000,1.5,Low Vision
...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,SV212,15,Averaging,mll,49.010567,103.626936,0.826495,1.0,0.201895,0.2,4000,1.5,Control
226,SV213,15,Averaging,mll,20.705631,47.017065,0.996802,1.0,0.349268,0.2,4000,1.5,Control
227,SV214,15,Averaging,mll,55.815413,117.236627,0.828403,1.0,0.201895,0.2,4000,1.5,Control
228,SV215,15,Averaging,mll,97.706203,201.018208,0.972679,1.0,0.394415,0.2,4000,1.5,Control


In [ ]:
from bcitoolbox_local import fit
import pandas as pd
import numpy as np
from helpers import get_subject_ids_for_beep


# Estimate sigma_a
subject_ids = get_subject_ids_for_beep()

#           pcommon, sigma_v, sigma_a, sigma_p, mu_p, dU, dD
es_para =  [0,       0,       1,       0,       0,    0,  0]
fixvalue = [0,       4000,     0.2,     4000,    0,    0,  0]

fitting_results = []
output_path = 'csv/modeling/outputs/sigma_a_summary.csv'
for subject in subject_ids:
    file_name = f"{subject}_beep.csv"

    data_file_path = 'csv/modeling/data/' + file_name
    behavior_data = np.loadtxt(data_file_path, delimiter=',')
    n_parameters, n_simulation =1, 10000

    estimated_parameters, error, strategy_name, bic, r2, fixvalue = fit(n_parameters, n_simulation, behavior_data,
                                                                        bounds=[(0.1,3)], es_para=es_para, fixvalue=fixvalue,
                                                                        Strategies=['sel'], FitType='mll')
    sigma_a = estimated_parameters[0]
    print("Estimated sigma_a:", sigma_a)

    row = {
        'subject_id': subject,
        'sigma_a': sigma_a.round(3),
        'error': error.round(3),
        'bic': bic.round(3),
        'r2': r2.round(3),
    }
    fitting_results.append(row)

# Save per-model results and pick best models
results_df = pd.DataFrame(fitting_results)
results_df.to_csv(output_path, index=False)

print(f"Mean estimated sigma_a = {results_df['sigma_a'].mean():.3f} +/- {results_df['sigma_a'].std():.3f}")